In [1]:
# # pip install accelerate

# from transformers import AutoProcessor, Gemma3ForConditionalGeneration
# from PIL import Image
# import requests
# import torch

# model_id = "google/gemma-3-4b-pt"

# url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
# image = Image.open(requests.get(url, stream=True).raw)

# model = Gemma3ForConditionalGeneration.from_pretrained(model_id).eval()
# processor = AutoProcessor.from_pretrained(model_id)

# prompt = "<start_of_image> in this image, there is"
# model_inputs = processor(text=prompt, images=image, return_tensors="pt")

# input_len = model_inputs["input_ids"].shape[-1]

# with torch.inference_mode():
#     generation = model.generate(**model_inputs, max_new_tokens=100, do_sample=False)
#     generation = generation[0][input_len:]

# decoded = processor.decode(generation, skip_special_tokens=True)
# print(decoded)


In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
import requests
import torch

# 自动选择设备（GPU 优先）
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

model_id = "google/gemma-3-4b-pt"

# 下载图片
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 🔧 配置 int4 量化参数
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 启用 int4 量化
    bnb_4bit_use_double_quant=True,         # 启用双量化（提升精度）
    bnb_4bit_quant_type="nf4",              # 使用 nf4（更稳定的量化类型）
    bnb_4bit_compute_dtype=torch.float16    # 推理计算使用 float16，兼顾速度与精度
)

# 加载模型（int4 量化）
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"  # 自动分配设备
).eval()

# 加载 processor（不需要放到 device 上）
processor = AutoProcessor.from_pretrained(model_id)

# 构造输入并转移到 device
prompt = "<start_of_image> in this image, there is"
model_inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

# 获取 prompt token 长度
input_len = model_inputs["input_ids"].shape[-1]

# 推理并生成
with torch.inference_mode():
    generation = model.generate(**model_inputs, max_new_tokens=100, do_sample=False)
    generation = generation[0][input_len:]

# 解码输出文本
decoded = processor.decode(generation, skip_special_tokens=True)
print("📝 Generated text:")
print(decoded)


: 

: 